# Acoustic · Photonic Measurement · Modern Physics · Torque/Force/Moment
# Linux Science Pipelines · Building Energy Creep · Biotech Singer Experiments

> **Repo:** `ColinsCoding/Dispersion-Assisted-GS-Phase-Recovery`  
> **File:** `notebooks/acoustic_photonic_biotech_singer_001.ipynb`

**Stack:** SymPy → NumPy → Torch → Matplotlib · subprocess (Linux)

§1 Acoustic + photonic measurement — photoacoustic, laser Doppler, interferometry  
§2 Modern physics — phonon dispersion, acoustic radiation pressure, quantum acoustics  
§3 Statics: torque, force, moment — vocal cord tension, tympanic membrane, torsion  
§4 Linux science pipelines — subprocess, Makefile, data pipeline class  
§5 Building energy creep — Kelvin-Voigt viscoelastic, thermal drift, creep ODE  
§6 Biotech singer experiments — glottal pulse, formants, jitter/shimmer, pitch detection

In [ ]:
import sympy as sp
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as scipy_signal
from scipy.io import wavfile
from scipy.fft import fft, fftfreq
import subprocess, os, sys, textwrap, warnings
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from IPython.display import display, Math, Code
warnings.filterwarnings('ignore')

sp.init_printing(use_latex='mathjax')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})
torch.set_default_dtype(torch.float64)

# physical constants
c_sound = 343.0        # m/s at 20°C
rho_air = 1.204        # kg/m³
c_light = 3e8          # m/s
hbar    = 1.0546e-34   # J·s
k_B     = 1.3806e-23   # J/K
print('ready')

---
# §1 — Acoustic + Photonic Measurement

## 1a. Photoacoustic Spectroscopy (PAS)
Modulated laser → absorbed by sample → thermal expansion → acoustic wave detected by microphone.  
$$p_{\\text{PA}}(\\omega) = \\frac{(\\gamma-1)}{V} \\cdot \\alpha(\\lambda) \\cdot P_0 \\cdot H(\\omega)$$

## 1b. Laser Doppler Vibrometry (LDV)
Surface velocity from Doppler shift: $\\Delta f = 2v/\\lambda$.  
Detected by homodyne/heterodyne interferometer — used on vocal cords, eardrums, buildings.

## 1c. Connection to Dispersion-Assisted GS
The DFT chirped-pulse technique from the repo extends naturally:  
time-stretch the acoustic impulse response → measure in frequency domain → GS phase recovery.

In [ ]:
# ── SymPy: photoacoustic signal model ────────────────────────────
omega_s, alpha_s, P0_s, gamma_s, V_s = sp.symbols(
    'omega alpha P_0 gamma V', positive=True)
tau_s, L_s = sp.symbols('tau L', positive=True)

# Thermal diffusion length: μ = sqrt(2κ/(ρCω))
kappa_s, rho_s, Cp_s = sp.symbols('kappa rho C_p', positive=True)
mu_thermal = sp.sqrt(2*kappa_s / (rho_s * Cp_s * omega_s))
print('Thermal diffusion length μ(ω):')
display(Math(r'\mu(\omega) = \sqrt{\frac{2\kappa}{\rho C_p \omega}} = ' +
             sp.latex(mu_thermal)))

# PA signal: thermally thin vs thick
# Thermally thin: L << μ → signal ∝ 1/ω
# Thermally thick: L >> μ → signal ∝ 1/ω^(3/2)
H_thin  = alpha_s * P0_s * (gamma_s - 1) / (V_s * omega_s)
H_thick = alpha_s * P0_s * (gamma_s - 1) / (V_s * omega_s**sp.Rational(3,2))

print('\nPA signal (thermally thin vs thick):')
display(Math(r'p_{\text{thin}} = ' + sp.latex(H_thin)))
display(Math(r'p_{\text{thick}} = ' + sp.latex(H_thick)))

# ── Numerical: PA spectrum simulation ─────────────────────────────
freqs_pa = np.logspace(1, 5, 500)   # 10 Hz to 100 kHz
omega_pa = 2*np.pi*freqs_pa

# absorption lines of common bio-molecules (simulated)
# CO₂ at 4.26 μm, H₂O at 6.0 μm, protein amide at ~6 μm
bio_molecules = [
    ('CO₂  (breath)',    0.3,  [50, 200, 800]),
    ('H₂O  (hydration)', 0.15, [100, 400, 1500]),
    ('NH₃  (metabolism)',0.08, [300, 1200]),
    ('Protein (tissue)', 0.05, [600, 2000, 8000]),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (name, absorb, res_freqs) in zip(axes.flat, bio_molecules):  # loop
    # PA response: thin regime with resonances
    H = absorb / omega_pa
    for f0 in res_freqs:          # loop over resonances
        Q = 30; w0 = 2*np.pi*f0
        H *= np.abs(w0**2 / (w0**2 - omega_pa**2 + 1j*omega_pa*w0/Q))
    ax.loglog(freqs_pa, np.abs(H), 'b-', lw=1.8)
    for f0 in res_freqs:
        ax.axvline(f0, ls='--', color='red', alpha=0.5, lw=0.8)
    ax.set_title(name); ax.set_xlabel('f (Hz)'); ax.set_ylabel('|H(ω)|')
    ax.grid(True, alpha=0.3, which='both')
plt.suptitle('Photoacoustic spectra — bio-molecule resonances', fontweight='bold')
plt.tight_layout(); plt.show()

# ── Laser Doppler Vibrometry: Doppler shift ────────────────────────
print('\nLDV: Doppler frequency shift Δf = 2v·cos(θ)/λ')
lam_vals = [532e-9, 633e-9, 1064e-9, 1550e-9]   # green, HeNe, Nd:YAG, telecom
v_range  = np.linspace(0, 5, 200)                  # vocal cord velocity 0–5 mm/s

fig, ax = plt.subplots(figsize=(9, 4))
for lam in lam_vals:                              # loop over laser wavelengths
    df = 2 * v_range * 1e-3 / lam
    ax.plot(v_range, df*1e-3, lw=2, label=f'λ={lam*1e9:.0f}nm')
ax.set_xlabel('Surface velocity (mm/s)'); ax.set_ylabel('Doppler shift (kHz)')
ax.set_title('LDV Doppler shift: vocal cord velocity range')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# ── Interferometric SNR: shot noise limit ─────────────────────────
print('\nInterferometric acoustic measurement — shot noise SNR:')
P_probe  = np.logspace(-6, -1, 100)   # 1 μW to 100 mW
BW_det   = 100e3                       # 100 kHz bandwidth
e_charge = 1.6e-19; eta_det = 0.7
hf_633   = 6.626e-34 * 3e8 / 633e-9   # photon energy
i_shot   = np.sqrt(2 * e_charge * eta_det * P_probe / hf_633 * BW_det)
i_signal = eta_det * P_probe / hf_633 * 1e-9   # 1 nm displacement at 633 nm
SNR_shot = 20*np.log10(i_signal / (i_shot + 1e-30))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.semilogx(P_probe*1e3, SNR_shot, 'g-', lw=2)
ax.axhline(0, ls='--', color='red', lw=1, label='SNR=0 dB')
ax.axhline(20, ls=':', color='blue', lw=1, label='20 dB margin')
ax.set_xlabel('Probe power (mW)'); ax.set_ylabel('Shot-noise SNR (dB)')
ax.set_title('LDV shot-noise limited SNR (BW=100 kHz, η=0.7, δx=1nm)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

---
# §2 — Modern Physics: Phonon Dispersion, Acoustic Radiation Pressure, Quantum Acoustics

**Phonon:** quantum of acoustic vibration. Dispersion relation connects ω and k.

$$E_n = \\hbar\\omega\\left(n + \\frac{1}{2}\\right), \\quad \\langle x^2 \\rangle = \\frac{\\hbar}{2m\\omega}(2n+1)$$

**Acoustic radiation pressure (Langevin):**  
$$P_{\\text{rad}} = \\frac{2\\langle E \\rangle}{V}$$  
Used for acoustic levitation, cell manipulation, vocal cord therapy.

In [ ]:
# ── SymPy: phonon zero-point motion ─────────────────────────────
n_s, m_s, om_s = sp.symbols('n m omega', positive=True)

# Zero-point fluctuation: <x²> = ħ/(2mω) for n=0
x2_zpf = hbar / (2 * m_s * om_s)
x_zpf  = sp.sqrt(x2_zpf)
print('Zero-point fluctuation (ZPF):')
display(Math(r'x_{\text{ZPF}} = \sqrt{\frac{\hbar}{2m\omega}} = ' + sp.latex(x_zpf)))

# Thermal occupation: n_BE = 1/(exp(ħω/kT)-1)
T_s, k_s = sp.symbols('T k_B', positive=True)
n_BE = 1 / (sp.exp(hbar*om_s/(k_s*T_s)) - 1)
# High-T limit (classical)
n_BE_hT = sp.series(n_BE, sp.Symbol('dummy'))

# ── Numerical: phonon dispersion relations ────────────────────────
# 1. Monatomic linear chain: ω = 2√(κ/m)|sin(ka/2)|
# 2. Diatomic chain: acoustic + optical branches
# 3. Debye model: linear ω = v_s·k
# 4. Einstein model: flat ω = ω_E
N_k = 500
k_arr = np.linspace(-np.pi, np.pi, N_k)  # in units of 1/a

kappa_spring = 1.0; m1 = 1.0; m2 = 2.0

# Monatomic
omega_mono = 2*np.sqrt(kappa_spring/m1) * np.abs(np.sin(k_arr/2))

# Diatomic: two branches
M = m1 + m2
omega_diatomic_ac  = np.sqrt(kappa_spring*M/(m1*m2) * (1 - np.sqrt(1 - 4*m1*m2/M**2 * np.sin(k_arr/2)**2)))
omega_diatomic_opt = np.sqrt(kappa_spring*M/(m1*m2) * (1 + np.sqrt(1 - 4*m1*m2/M**2 * np.sin(k_arr/2)**2)))

# Debye
v_debye = 1.5
omega_debye = v_debye * np.abs(k_arr)

# Zero-point motion vs frequency for bio-relevant masses
freqs_zpf = np.logspace(1, 9, 200)  # 10 Hz to 1 GHz
omega_zpf = 2*np.pi*freqs_zpf

bio_masses = [
    ('vocal cord (1g)',   1e-3),
    ('eardrum (10mg)',    10e-6),
    ('cell membrane (1ng)', 1e-12),
    ('protein molecule (100kDa)', 100e3 * 1.66e-27),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# phonon dispersion
axes[0].plot(k_arr/np.pi, omega_mono, 'b-', lw=2, label='Monatomic')
axes[0].plot(k_arr/np.pi, omega_diatomic_ac,  'g-', lw=2, label='Diatomic (acoustic)')
axes[0].plot(k_arr/np.pi, omega_diatomic_opt, 'r-', lw=2, label='Diatomic (optical)')
axes[0].plot(k_arr/np.pi, omega_debye, 'k--', lw=1, alpha=0.5, label='Debye')
axes[0].set_xlabel('k / (π/a)'); axes[0].set_ylabel('ω (a.u.)')
axes[0].set_title('Phonon dispersion relations'); axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# ZPF vs frequency for bio masses
for name, mass in bio_masses:       # loop over masses
    xzpf = np.sqrt(hbar / (2*mass*omega_zpf)) * 1e9   # nm
    axes[1].loglog(freqs_zpf, xzpf, lw=2, label=name)
axes[1].axhline(0.001, ls='--', color='gray', label='1 pm (atomic scale)')
axes[1].axvline(100, ls=':', color='red', lw=1, label='100 Hz voice')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('ZPF x_zpf (nm)')
axes[1].set_title('Zero-point fluctuation vs frequency'); axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, which='both')

# Acoustic radiation pressure: P_rad = E_density = p_rms²/(ρc²)
SPL_dB = np.linspace(60, 180, 200)   # dB SPL
p_rms  = 20e-6 * 10**(SPL_dB/20)    # Pa
P_rad  = p_rms**2 / (rho_air * c_sound**2)

axes[2].semilogy(SPL_dB, P_rad, 'purple', lw=2)
levels = {
    'Conversation (65 dB)': 65, 'Concert (110 dB)': 110,
    'Ultrasound therapy (150 dB)': 150, 'Acoustic levitation (160 dB)': 160,
}
for label_r, spl in levels.items():
    axes[2].axvline(spl, ls='--', alpha=0.6, lw=1)
    axes[2].text(spl+0.5, P_rad.min()*10, label_r, rotation=90, fontsize=7, va='bottom')
axes[2].set_xlabel('SPL (dB)'); axes[2].set_ylabel('Radiation pressure (Pa)')
axes[2].set_title('Acoustic radiation pressure vs SPL')
axes[2].grid(True, alpha=0.3)
plt.suptitle('Modern Physics: Phonons, ZPF, Acoustic Radiation', fontweight='bold')
plt.tight_layout(); plt.show()

# Bose-Einstein occupation vs temperature
T_vals   = np.array([1, 4, 77, 300])   # K
f_phonon = np.logspace(-1, 5, 300)     # GHz
print('\nMean phonon occupation n_BE at biological frequencies:')
print(f'{"T(K)":6}  ', end='')
for f in [0.1, 1.0, 10.0, 100.0, 1000.0]:
    print(f'{f:>10.1f}GHz ', end='')
print()
for T in T_vals:                        # loop over temperatures
    print(f'{T:6.1f}  ', end='')
    for f in [0.1, 1.0, 10.0, 100.0, 1000.0]:   # loop over frequencies
        om = 2*np.pi*f*1e9
        n  = 1/(np.exp(hbar*om/(k_B*T)) - 1) if hbar*om/(k_B*T) < 500 else 0
        print(f'{n:>12.2e} ', end='')
    print()

---
# §3 — Statics: Torque, Force, Moment

**Applied to acoustic bio-mechanics:**
- Vocal cord tension: string under distributed load, natural frequency $f_0 = \\frac{1}{2L}\\sqrt{T/\\mu}$
- Tympanic membrane: circular plate static deflection under sound pressure
- Cochlear basilar membrane: tapered beam stiffness gradient

**Torque:** $\\boldsymbol{\\tau} = \\mathbf{r} \\times \\mathbf{F}$, moment arm matters.

In [ ]:
# ── SymPy: vocal cord as tensioned string ────────────────────────
T_s, mu_s, L_v, n_s_vc = sp.symbols('T mu L n', positive=True)
x_s = sp.Symbol('x')

# Natural frequencies of tensioned string
f_n = n_s_vc / (2*L_v) * sp.sqrt(T_s / mu_s)
print('Vocal cord natural frequency (tensioned string):')
display(Math(r'f_n = \frac{n}{2L}\sqrt{\frac{T}{\mu}} = ' + sp.latex(f_n)))

# Wave equation for string: y_tt = (T/μ) y_xx
y_func = sp.Function('y')(x_s, sp.Symbol('t'))
c_str  = sp.sqrt(T_s/mu_s)
wave_eq = sp.Eq(y_func.diff(sp.Symbol('t'),2),
                c_str**2 * y_func.diff(x_s,2))
print('\nWave equation:')
display(Math(r'\frac{\partial^2 y}{\partial t^2} = \frac{T}{\mu}\frac{\partial^2 y}{\partial x^2}'))

# Torque balance on arytenoid cartilage (rotates to change cord tension)
F_crico, r_crico, F_thyro, r_thyro = sp.symbols('F_ct r_ct F_th r_th', positive=True)
# ΣM = 0 at arytenoid joint
tau_eq = sp.Eq(F_crico * r_crico, F_thyro * r_thyro)
T_tension = sp.solve(tau_eq, F_crico)[0]
print('\nTorque balance on arytenoid (cricothyroid force to produce tension):')
display(Math(r'F_{ct} = ' + sp.latex(T_tension)))

# ── Numerical: f0 as function of tension and length ───────────────
L_m    = np.linspace(10e-3, 25e-3, 200)  # vocal cord length 10-25 mm
T_vals = np.array([0.05, 0.1, 0.2, 0.5, 1.0])  # N
mu_v   = 0.3e-3   # kg/m (vocal cord linear density)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for T_val in T_vals:              # loop over tensions
    f0 = 1/(2*L_m) * np.sqrt(T_val/mu_v)
    axes[0].plot(L_m*1e3, f0, lw=2, label=f'T={T_val}N')
axes[0].axhspan(80, 350, alpha=0.15, color='green', label='Speaking range')
axes[0].axhspan(200, 1100, alpha=0.1, color='blue', label='Singing soprano')
axes[0].set_xlabel('Cord length (mm)'); axes[0].set_ylabel('f₀ (Hz)')
axes[0].set_title('Vocal cord fundamental frequency vs length'); axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Tympanic membrane: circular plate deflection under sound pressure
# Center deflection: w_max = p·a⁴/(64·D) for clamped circular plate
# D = E·t³/(12(1-ν²)) flexural rigidity
E_TM  = 10e6    # Pa Young's modulus (eardrum ~10 MPa)
nu_TM = 0.4     # Poisson ratio
t_TM  = 0.1e-3  # 0.1 mm thickness
a_TM  = 4e-3    # 4 mm radius
D_TM  = E_TM * t_TM**3 / (12*(1-nu_TM**2))

SPL_range = np.linspace(0, 120, 200)   # dB SPL
p_sound   = 20e-6 * 10**(SPL_range/20)
w_max     = p_sound * a_TM**4 / (64*D_TM)

axes[1].semilogy(SPL_range, w_max*1e9)
axes[1].axhline(0.1, ls='--', color='orange', label='0.1 nm (LDV limit)')
axes[1].axhline(1.0, ls='--', color='green',  label='1 nm')
axes[1].axhline(1e3, ls='--', color='red',    label='1 μm (pain threshold ~120dB)')
axes[1].set_xlabel('SPL (dB)'); axes[1].set_ylabel('Center deflection (nm)')
axes[1].set_title('Tympanic membrane static deflection (clamped plate)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# Cochlear basilar membrane: tapered beam frequency map (Greenwood)
x_bm = np.linspace(0, 35e-3, 500)   # 35 mm cochlea
# Greenwood function: f(x) = A*(10^(ax) - k)  where x in mm
A_g = 165.4; a_g = 0.06; k_g = 0.88
f_bm = A_g * (10**(a_g * x_bm*1e3) - k_g)

# Stiffness gradient: k(x) ∝ e^(-bx)
b_stiff = 2e3   # m⁻¹ stiffness decay
K_bm    = 3e-2 * np.exp(-b_stiff * x_bm)   # N/m

ax2b = axes[2].twinx()
axes[2].semilogy(x_bm*1e3, f_bm, 'b-', lw=2, label='CF (Greenwood)')
ax2b.semilogy(x_bm*1e3, K_bm, 'r-', lw=2, label='Stiffness k(x)')
axes[2].set_xlabel('Distance from base (mm)'); axes[2].set_ylabel('Characteristic freq (Hz)', color='b')
ax2b.set_ylabel('Stiffness k (N/m)', color='r')
axes[2].set_title('Cochlear basilar membrane tonotopy')
axes[2].grid(True, alpha=0.3)
plt.suptitle('Acoustic Bio-Mechanics: Vocal Cord · Eardrum · Cochlea', fontweight='bold')
plt.tight_layout(); plt.show()

# ── 3D torque calculation (Torch) ─────────────────────────────────
# Batch torque τ = r × F for cricoarytenoid joint
B_torq = 500
r_vecs = torch.randn(B_torq, 3, dtype=torch.float64)
r_vecs[:, 2] = 0   # 2D approximation: z=0
r_vecs = r_vecs / r_vecs.norm(dim=1, keepdim=True) * 0.005  # 5 mm arm
F_vecs = torch.zeros(B_torq, 3, dtype=torch.float64)
F_vecs[:, 0] = torch.randn(B_torq, dtype=torch.float64) * 0.5  # Fx
F_vecs[:, 1] = torch.abs(torch.randn(B_torq, dtype=torch.float64)) * 0.3  # Fy > 0

torques = torch.linalg.cross(r_vecs, F_vecs)   # (B, 3)
tau_z   = torques[:, 2]   # z-component (out-of-plane)
print(f'\nBatch torque magnitude |τ_z|: '
      f'mean={tau_z.abs().mean()*1e3:.3f} mN·m  '
      f'max={tau_z.abs().max()*1e3:.3f} mN·m')

---
# §4 — Linux Science Pipelines

**Pattern:** `data source → preprocessing → analysis → output` as a composable chain.  
Each stage is a subprocess or Python call; the pipeline class wires them together.

**Repo context:** these pipelines process raw experimental data → GS phase recovery → results.

In [ ]:
import subprocess, os, sys, shutil, textwrap
from pathlib import Path

# ── Makefile generator for the repo ──────────────────────────────
makefile_content = textwrap.dedent("""
    # Dispersion-Assisted-GS-Phase-Recovery Science Pipeline
    # Usage: make all  |  make clean  |  make env  |  make test

    PYTHON   := python3
    CONDA    := conda
    ENV_NAME := gs_phase
    NB_DIR   := notebooks
    SRC_DIR  := src
    DATA_DIR := data
    OUT_DIR  := results

    .PHONY: all env install test clean run-notebooks

    all: env install run-notebooks

    # Create conda environment
    env:
    \t$(CONDA) create -n $(ENV_NAME) python=3.11 -y
    \t$(CONDA) run -n $(ENV_NAME) pip install -r requirements.txt

    # Install package in editable mode
    install:
    \t$(PYTHON) -m pip install -e .

    # Run all notebooks headlessly
    run-notebooks:
    \t@for nb in $(NB_DIR)/*.ipynb; do \\
    \t\techo "Running $$nb ..."; \\
    \t\tjupyter nbconvert --to notebook --execute $$nb \\
    \t\t\t--output $$nb --ExecutePreprocessor.timeout=600; \\
    \tdone

    # GS phase recovery pipeline
    gs-recover:
    \t$(PYTHON) $(SRC_DIR)/gs_core.py \\
    \t\t--input  $(DATA_DIR)/raw_intensity.npy \\
    \t\t--output $(OUT_DIR)/recovered_phase.npy \\
    \t\t--n_iter 100 --D -5000

    # Acoustic measurement preprocessing
    preprocess-audio:
    \t$(PYTHON) -c "import sys; sys.path.insert(0,'$(SRC_DIR)'); \\
    \t\tfrom pipeline import preprocess_audio; \\
    \t\tpreprocess_audio('$(DATA_DIR)/audio', '$(OUT_DIR)/audio_processed')"

    test:
    \tpytest tests/ -v --tb=short

    clean:
    \trm -rf $(OUT_DIR)/*.npy $(OUT_DIR)/*.png
    \tfind . -name '__pycache__' -exec rm -rf {} +
    \tfind . -name '*.pyc' -delete
""".strip())

print('Generated Makefile:')
print(makefile_content)

# ── SciencePipeline class ─────────────────────────────────────────
from typing import Callable, Any

class SciencePipeline:
    """
    Composable science data pipeline.
    Stages run in sequence; each stage receives previous output.
    Supports subprocess stages (shell commands) and Python callables.
    """
    def __init__(self, name: str, verbose: bool = True):
        self.name    = name
        self.verbose = verbose
        self.stages: List[Dict] = []
        self.log:    List[str]  = []
        self.results: Dict      = {}

    def add_stage(self, name: str, fn: Callable, **kwargs):
        """Add a Python function stage."""
        self.stages.append({'type':'python', 'name':name, 'fn':fn, 'kwargs':kwargs})

    def add_shell(self, name: str, cmd: str):
        """Add a shell command stage."""
        self.stages.append({'type':'shell', 'name':name, 'cmd':cmd})

    def run(self, initial_data=None):
        """Execute all pipeline stages in order."""
        data = initial_data
        for stage in self.stages:          # loop over stages
            if self.verbose: print(f'  [{self.name}] → {stage["name"]}')
            if stage['type'] == 'python':
                data = stage['fn'](data, **stage['kwargs'])
                self.results[stage['name']] = data
            elif stage['type'] == 'shell':
                try:
                    result = subprocess.run(stage['cmd'], shell=True, capture_output=True,
                                            text=True, timeout=30)
                    self.log.append(f'{stage["name"]}: rc={result.returncode}')
                    if result.stdout: print(f'    stdout: {result.stdout.strip()[:100]}')
                    if result.returncode != 0:
                        print(f'    stderr: {result.stderr.strip()[:100]}')
                except subprocess.TimeoutExpired:
                    print(f'    TIMEOUT: {stage["name"]}')
        return data

    def summary(self):
        print(f'Pipeline: {self.name}  ({len(self.stages)} stages)')
        for i, s in enumerate(self.stages):  # loop
            print(f'  [{i+1:2d}] {s["type"]:8s} {s["name"]}')


# ── Acoustic measurement pipeline ─────────────────────────────────
def stage_load_audio(data, sr=22050, duration=2.0):
    """Simulate loading audio (in real use: scipy.io.wavfile.read)"""
    t = np.linspace(0, duration, int(sr*duration))
    # Simulated singing: F0=220 Hz (A3) + harmonics + noise
    f0 = 220.0
    signal = sum(1/n * np.sin(2*np.pi*n*f0*t) for n in range(1,8))
    signal += 0.02 * np.random.randn(len(t))
    return {'audio': signal, 'sr': sr, 't': t}

def stage_preemphasis(data, coeff=0.97):
    """Pre-emphasis filter: boost high frequencies."""
    from scipy.signal import lfilter
    audio_pe = lfilter([1, -coeff], [1], data['audio'])
    data['audio_pe'] = audio_pe
    return data

def stage_frame_analysis(data, frame_ms=25, hop_ms=10):
    """Split into overlapping frames for STFT."""
    sr = data['sr']
    frame_len = int(frame_ms/1000 * sr)
    hop_len   = int(hop_ms/1000  * sr)
    audio     = data['audio_pe']
    n_frames  = (len(audio) - frame_len) // hop_len + 1
    frames    = np.zeros((n_frames, frame_len))
    for i in range(n_frames):    # loop: frame extraction
        frames[i] = audio[i*hop_len : i*hop_len+frame_len] * np.hanning(frame_len)
    data['frames'] = frames; data['frame_len'] = frame_len; data['hop_len'] = hop_len
    return data

def stage_spectrum(data):
    """Compute short-time magnitude spectrum."""
    NFFT = 512
    spec = np.abs(np.fft.rfft(data['frames'], n=NFFT))**2
    data['spectrum'] = spec
    data['freqs']    = np.fft.rfftfreq(NFFT, 1/data['sr'])
    return data

def stage_gs_phase_recovery(data, n_iter=50):
    """Apply GS phase recovery to each spectral frame."""
    spec = data['spectrum']
    A_target = np.sqrt(spec + 1e-30)
    n_frames, n_bins = A_target.shape
    phase_rec = np.zeros_like(A_target)

    for fi in range(n_frames):          # loop over frames
        phi = 2*np.pi*np.random.rand(n_bins)
        E   = A_target[fi] * np.exp(1j*phi)
        for _ in range(n_iter):         # GS iterations
            E   = A_target[fi] * np.exp(1j*np.angle(E))
            e_t = np.fft.irfft(E)
            E   = np.fft.rfft(e_t)
        phase_rec[fi] = np.angle(E)

    data['phase_recovered'] = phase_rec
    return data

# ── Run the pipeline ───────────────────────────────────────────────
pipe = SciencePipeline('acoustic-gs-pipeline')
pipe.add_stage('load_audio',        stage_load_audio)
pipe.add_stage('preemphasis',       stage_preemphasis)
pipe.add_stage('frame_analysis',    stage_frame_analysis)
pipe.add_stage('spectrum',          stage_spectrum)
pipe.add_stage('gs_phase_recovery', stage_gs_phase_recovery, n_iter=30)
pipe.add_shell('git_status',        'git status --short 2>/dev/null || echo "not a git repo"')
pipe.add_shell('python_version',    f'{sys.executable} --version')
pipe.add_shell('disk_usage',        'df -h . 2>/dev/null || echo "df not available"')

pipe.summary()
print(); result = pipe.run()

In [ ]:
# ── Visualize pipeline outputs ────────────────────────────────────
res = result
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Raw waveform
axes[0,0].plot(res['t'][:2000], res['audio'][:2000], 'b-', lw=0.8)
axes[0,0].set_title('Raw waveform (A3 singing)'); axes[0,0].set_xlabel('t (s)')

# Pre-emphasized
axes[0,1].plot(res['t'][:2000], res['audio_pe'][:2000], 'g-', lw=0.8)
axes[0,1].set_title('Pre-emphasized (α=0.97)')

# Spectrogram
spec_dB = 10*np.log10(res['spectrum'].T + 1e-30)
axes[0,2].imshow(spec_dB, aspect='auto', origin='lower', cmap='magma',
                  extent=[0, res['t'][-1], 0, res['sr']/2/1000])
axes[0,2].set_xlabel('Time (s)'); axes[0,2].set_ylabel('Freq (kHz)')
axes[0,2].set_title('Spectrogram (STFT)')

# Recovered phase
axes[1,0].imshow(res['phase_recovered'].T, aspect='auto', origin='lower',
                  cmap='RdBu', extent=[0, res['t'][-1], 0, res['sr']/2/1000])
axes[1,0].set_xlabel('Time (s)'); axes[1,0].set_ylabel('Freq (kHz)')
axes[1,0].set_title('GS-recovered phase (rad)')

# Mean spectrum
mean_spec = res['spectrum'].mean(axis=0)
axes[1,1].semilogy(res['freqs']/1000, mean_spec, 'b-', lw=1.5)
# mark harmonics
for harmonic in range(1, 9):
    axes[1,1].axvline(220*harmonic/1000, ls='--', color='red', alpha=0.5, lw=0.8)
axes[1,1].set_xlabel('Freq (kHz)'); axes[1,1].set_title('Mean spectrum + harmonic markers')
axes[1,1].grid(True, alpha=0.3)

# Pipeline stage log
axes[1,2].axis('off')
log_text = 'Pipeline log:\n' + '\n'.join(pipe.log)
axes[1,2].text(0.05, 0.95, log_text, transform=axes[1,2].transAxes,
               va='top', ha='left', fontsize=9, fontfamily='monospace')
axes[1,2].set_title('Shell stage log')

plt.suptitle('Science Pipeline: Raw Audio → GS Phase Recovery', fontweight='bold', fontsize=12)
plt.tight_layout(); plt.show()

---
# §5 — Building Energy Creep: Viscoelastic Model + Thermal Drift

**Creep:** time-dependent deformation under constant load. In buildings:  
- Concrete creep: sustained load → slow deformation over years  
- Thermal drift: gradual energy state change from day/night cycling  
- Sensor drift: photonic/acoustic sensors degrade over time

**Kelvin-Voigt model:** spring $E$ in parallel with dashpot $\\eta$:  
$$\\sigma = E\\varepsilon + \\eta\\dot{\\varepsilon} \\implies \\varepsilon(t) = \\frac{\\sigma_0}{E}\\left(1 - e^{-Et/\\eta}\\right)$$

In [ ]:
# ── SymPy: Kelvin-Voigt creep solution ───────────────────────────
eps_s = sp.Function('varepsilon')(sp.Symbol('t'))
E_mod, eta_s, sigma0_s, t_sym = sp.symbols('E eta sigma_0 t', positive=True)

# ODE: E*ε + η*dε/dt = σ₀
creep_ode = sp.Eq(E_mod*eps_s + eta_s*eps_s.diff(t_sym), sigma0_s)
creep_sol = sp.dsolve(creep_ode, eps_s,
                      ics={eps_s.subs(t_sym,0): 0})
print('Kelvin-Voigt creep ODE solution:')
display(Math(sp.latex(creep_sol)))

# Maxwell model: spring + dashpot in series → stress relaxation
sigma_s = sp.Function('sigma')(t_sym)
relax_ode = sp.Eq(sigma_s.diff(t_sym) + E_mod/eta_s * sigma_s, 0)
relax_sol = sp.dsolve(relax_ode, sigma_s,
                      ics={sigma_s.subs(t_sym,0): sigma0_s})
print('\nMaxwell relaxation solution:')
display(Math(sp.latex(relax_sol)))

# ── Numerical: building energy creep scenarios ────────────────────
# Time axis: 0 to 50 years in hours
t_years = np.linspace(0, 50, 5000)
t_hours = t_years * 8760

# Creep coefficient model (CEB-FIP): φ(t) = φ_∞ * β_c(t)
# β_c(t) = ((t-t0)/T_h)^0.3 / (1 + ((t-t0)/T_h)^0.3)
phi_inf = 2.5    # ultimate creep coefficient (normal concrete)
T_h     = 1000   # characteristic time (hours)
t0_h    = 720    # loading age (1 month in hours)

# Loop over loading ages and concrete grades
loading_scenarios = [
    ('Loaded at 28 days (normal)',  720,   2.5),
    ('Loaded at 7 days (early)',    168,   3.2),
    ('Loaded at 90 days (late)',    2160,  1.8),
    ('High-strength concrete',      720,   1.4),
    ('Lightweight concrete',        720,   3.0),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for name, t0, phi_inf_v in loading_scenarios:   # loop over scenarios
    mask = t_hours > t0
    dt_h = t_hours[mask] - t0
    beta_c = (dt_h/T_h)**0.3 / (1 + (dt_h/T_h)**0.3)
    phi_t  = phi_inf_v * beta_c
    eps_e  = 1e-3         # elastic strain (concrete under 10 MPa)
    eps_cr = eps_e * phi_t  # creep strain
    axes[0,0].plot(t_years[mask], eps_cr*1e3, lw=2, label=name)

axes[0,0].set_xlabel('Time (years)'); axes[0,0].set_ylabel('Creep strain (mm/m)')
axes[0,0].set_title('Concrete creep strain vs time (CEB-FIP model)')
axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3)

# Thermal drift in building energy model
# E(t) = E_base + ΔE*sin(2πt/24) + drift*t  (daily cycle + long-term drift)
t_days = np.linspace(0, 365, 8760)   # 1 year in hours
E_base = 50.0   # kWh/day baseline
E_seasonal = E_base + 20*np.sin(2*np.pi*t_days/365 + np.pi)  # winter peak
E_diurnal  = 15*np.sin(2*np.pi*t_days*24/365)   # daily cycle
E_creep    = 0.003 * t_days                       # slow drift (aging)
E_total    = E_seasonal + E_diurnal + E_creep + 2*np.random.randn(8760)

axes[0,1].plot(t_days, E_total, 'b-', lw=0.5, alpha=0.6, label='Total')
# 7-day moving average
E_smooth = np.convolve(E_total, np.ones(168)/168, mode='same')
axes[0,1].plot(t_days, E_smooth, 'r-', lw=2, label='7-day MA')
axes[0,1].plot(t_days, E_seasonal + E_creep, 'g--', lw=1.5, label='Seasonal+drift')
axes[0,1].set_xlabel('Day of year'); axes[0,1].set_ylabel('Energy (kWh/day)')
axes[0,1].set_title('Building energy: seasonal + diurnal + creep drift')
axes[0,1].legend(fontsize=9); axes[0,1].grid(True, alpha=0.3)

# Sensor drift model: photonic sensor ZPD (zero path difference) drift
# Relevant to interferometric acoustic measurement in repo context
t_sensor = np.linspace(0, 365*5, 10000)   # 5 years
# Drift models: linear, Arrhenius (thermal activation), random walk
drift_models = [
    ('Linear aging',          0.001 * t_sensor,                    'b-'),
    ('Arrhenius (T=25°C)',    0.0005 * t_sensor**0.7,              'g-'),
    ('Random walk',           np.cumsum(0.003*np.random.randn(len(t_sensor)))*np.sqrt(1), 'orange'),
    ('Kelvin-Voigt creep',    2*(1 - np.exp(-t_sensor/500)),       'r-'),
]
for name, drift, style in drift_models:    # loop over drift models
    axes[1,0].plot(t_sensor/365, drift, style, lw=1.8, label=name)
axes[1,0].axhline(0.5, ls='--', color='red', label='Recalibration threshold')
axes[1,0].set_xlabel('Time (years)'); axes[1,0].set_ylabel('Sensor drift (nm ZPD)')
axes[1,0].set_title('Photonic sensor drift models over 5 years')
axes[1,0].legend(fontsize=8); axes[1,0].grid(True, alpha=0.3)

# Viscoelastic spring-dashpot Torch batch
E_vals  = torch.logspace(-1, 2, 20, dtype=torch.float64)  # 0.1-100 GPa
eta_vals= torch.logspace(2,  6, 20, dtype=torch.float64)  # 100-1M GPa·s
EE, ETA = torch.meshgrid(E_vals, eta_vals, indexing='ij')
tau_relax = ETA / EE                                       # relaxation time τ = η/E
t_probe   = torch.tensor(3650.0 * 24, dtype=torch.float64)  # 10 years in hours
creep_frac = 1 - torch.exp(-t_probe / tau_relax)           # Kelvin-Voigt creep fraction

im = axes[1,1].contourf(np.log10(E_vals.numpy()), np.log10(eta_vals.numpy()),
                          creep_frac.numpy().T, levels=20, cmap='RdYlGn_r')
axes[1,1].contour(np.log10(E_vals.numpy()), np.log10(eta_vals.numpy()),
                   creep_frac.numpy().T, levels=[0.5, 0.9], colors='white', linewidths=1.5)
plt.colorbar(im, ax=axes[1,1], label='Creep fraction at 10 years')
axes[1,1].set_xlabel('log₁₀(E [GPa])'); axes[1,1].set_ylabel('log₁₀(η [GPa·s])')
axes[1,1].set_title('Kelvin-Voigt: creep fraction (Torch batch)')

plt.suptitle('Building Energy Creep: Concrete · Thermal Drift · Sensor Aging', fontweight='bold')
plt.tight_layout(); plt.show()

---
# §6 — Biotech Singer Experiments

## Experiments to try:

| # | Experiment | Measurement | What you learn |
|---|-----------|-------------|----------------|
| 1 | Record sustained vowel /aː/ at 3 pitches | f₀, harmonics, HNR | Vocal health, breathiness |
| 2 | Glottal inverse filtering | Glottal flow waveform | Open/closed quotient |
| 3 | Formant tracking across vowels | F1/F2/F3 trajectories | Vocal tract shape |
| 4 | Jitter/Shimmer analysis | Cycle-to-cycle variation | Vocal fold pathology |
| 5 | Photoacoustic laryngoscopy | Sub-glottal pressure | Real-time vocal effort |
| 6 | LDV on neck surface | Vocal cord vibration | Non-invasive displacement |
| 7 | Dispersion-assisted measurement | Chirped acoustic pulse | Room acoustics + RT60 |

In [ ]:
# ── Glottal pulse models ─────────────────────────────────────────
# LF model (Liljencrants-Fant): physically motivated glottal flow derivative

def lf_model(t, Ee=1.0, tp=0.35, te=0.80, ta=0.10, f0=220):
    """
    LF model of glottal flow derivative dU/dt.
    t: time array [0, 1/f0]
    tp: time of peak opening phase (fraction of T0)
    te: time of abrupt closure (fraction of T0)
    ta: return phase duration (fraction of T0)
    """
    T0   = 1.0/f0
    t_n  = t % T0 / T0   # normalized 0-1
    E0   = -Ee / np.sin(np.pi * te)
    alpha= np.log(Ee/1e-6) / (tp - te) if tp != te else 1.0
    dU   = np.zeros_like(t)

    for i, tn in enumerate(t_n):    # loop over samples
        if tn <= te:
            dU[i] = E0 * np.exp(alpha*(tn-te)) * np.sin(np.pi*tn/te) if te>0 else 0
        else:
            eps_a = 1/(ta + 1e-10)
            dU[i] = -Ee/((te-1)*ta) * (np.exp(-eps_a*(tn-te)) - np.exp(-eps_a*(1-te)))
    return dU

# ── Vocal tract filter (formant synthesis) ───────────────────────
def formant_filter(f0, formants, bandwidths, n_pts=8192, sr=22050):
    """
    Build vocal tract transfer function from formant frequencies and bandwidths.
    H(s) = ∏ ωₙ² / (s² + 2ζωₙs + ωₙ²)
    Returns: freq array, |H(f)|
    """
    freqs = np.fft.rfftfreq(n_pts, 1/sr)
    H = np.ones(len(freqs), dtype=complex)
    for fn, bw in zip(formants, bandwidths):   # loop over formants
        s  = 2j*np.pi*freqs
        wn = 2*np.pi*fn
        zeta = np.pi*bw/wn
        H *= wn**2 / (s**2 + 2*zeta*wn*s + wn**2)
    return freqs, np.abs(H)

# ── Jitter & Shimmer analysis ─────────────────────────────────────
def compute_jitter_shimmer(audio, sr, f0_est):
    """
    Estimate jitter (cycle-to-cycle period variation) and
    shimmer (amplitude variation) from a voice sample.
    """
    # Find glottal cycles by peak picking
    T0_samples = int(sr / f0_est)
    peaks = []
    search_start = T0_samples // 2
    for k in range(0, len(audio) - T0_samples, T0_samples // 2):  # loop
        seg = audio[k:k+T0_samples]
        if len(seg) > 0:
            peaks.append(k + np.argmax(np.abs(seg)))

    if len(peaks) < 3:
        return {'jitter_pct': np.nan, 'shimmer_pct': np.nan}

    # Period sequence
    periods = np.diff(peaks).astype(float) / sr   # in seconds
    amps    = np.array([np.max(np.abs(audio[p:p+T0_samples])) for p in peaks[:-1]])

    # Jitter: relative average perturbation (RAP)
    jitter_abs = np.mean(np.abs(np.diff(periods)))
    jitter_pct = 100 * jitter_abs / np.mean(periods)

    # Shimmer: dB
    shimmer_dB = np.mean(np.abs(20*np.log10(amps[1:]/amps[:-1] + 1e-9)))

    # HNR: harmonics-to-noise ratio
    fft_a = np.abs(np.fft.rfft(audio, n=8192))
    freqs_a = np.fft.rfftfreq(8192, 1/sr)
    harmonic_power = 0; noise_power = 0
    for h in range(1, 12):          # loop over harmonics
        fc = h * f0_est
        bw = f0_est * 0.3
        mask_h = np.abs(freqs_a - fc) < bw
        mask_n = (np.abs(freqs_a - fc) >= bw) & (np.abs(freqs_a - fc) < 3*bw)
        harmonic_power += fft_a[mask_h].sum()**2
        noise_power    += fft_a[mask_n].sum()**2
    HNR = 10*np.log10(harmonic_power / (noise_power + 1e-30))

    return {'jitter_pct': jitter_pct, 'shimmer_dB': shimmer_dB, 'HNR_dB': HNR,
            'periods': periods, 'amps': amps}


# ── Simulate singer experiments ────────────────────────────────────
sr_voice = 22050
dur      = 2.0
t_voice  = np.linspace(0, dur, int(sr_voice*dur))

# Vowel formant database (F1, F2, F3 in Hz)
vowel_formants = {
    '/aː/ (soprano, A4=440Hz)':  ([700, 1100, 2600], [80, 90, 120]),
    '/eː/ (soprano, A4=440Hz)':  ([400, 2300, 2900], [60, 100, 130]),
    '/iː/ (soprano, A4=440Hz)':  ([270, 2200, 3000], [50, 90, 120]),
    '/oː/ (baritone, A3=220Hz)': ([450,  800, 2500], [70, 80, 100]),
    '/uː/ (baritone, A3=220Hz)': ([300,  700, 2500], [60, 80, 110]),
}

f0_map = {'/aː/ (soprano, A4=440Hz)': 440, '/eː/ (soprano, A4=440Hz)': 440,
           '/iː/ (soprano, A4=440Hz)': 440, '/oː/ (baritone, A3=220Hz)': 220,
           '/uː/ (baritone, A3=220Hz)': 220}

fig = plt.figure(figsize=(16, 12))
gs_fig = gridspec.GridSpec(3, 5, figure=fig, hspace=0.5, wspace=0.4)

biomarkers = []

for col, (vowel, (formants, bws)) in enumerate(vowel_formants.items()):  # loop: vowels
    f0_v = f0_map[vowel]

    # Generate glottal source
    dU  = lf_model(t_voice, f0=f0_v, tp=0.3, te=0.7)
    # Integrate to get glottal flow
    U   = np.cumsum(dU) / sr_voice
    # Convolve with vocal tract filter
    freqs_vt, H_vt = formant_filter(f0_v, formants, bws, n_pts=len(t_voice), sr=sr_voice)
    U_f   = np.fft.rfft(U)
    # Pad H_vt to match
    H_pad = H_vt[:len(U_f)]
    voice = np.fft.irfft(U_f * H_pad)
    voice = voice / (np.max(np.abs(voice)) + 1e-9)  # normalize
    # Add jitter/shimmer (realistic perturbation)
    jitter_noise = 0.02 * np.random.randn(len(voice))
    voice += jitter_noise

    # Plot glottal flow waveform (top row)
    ax_w = fig.add_subplot(gs_fig[0, col])
    ax_w.plot(t_voice[:int(5/f0_v*sr_voice)],
              voice[:int(5/f0_v*sr_voice)], 'b-', lw=1.5)
    ax_w.set_title(vowel.split('(')[0].strip(), fontsize=8)
    ax_w.set_xlabel('t(s)', fontsize=7); ax_w.tick_params(labelsize=7)

    # Plot vocal tract filter (middle row)
    ax_h = fig.add_subplot(gs_fig[1, col])
    ax_h.semilogy(freqs_vt/1000, H_vt, 'g-', lw=1.2)
    ax_h.set_xlim(0, 5)
    for fi, fn in enumerate(formants):   # loop: mark formants
        ax_h.axvline(fn/1000, ls='--', color='red', lw=0.8, alpha=0.7)
        ax_h.text(fn/1000+0.05, H_vt.max()*0.3, f'F{fi+1}={fn}', fontsize=6, color='red')
    ax_h.set_xlabel('Freq (kHz)', fontsize=7); ax_h.tick_params(labelsize=7)

    # Jitter/shimmer (bottom row text)
    bm = compute_jitter_shimmer(voice, sr_voice, f0_v)
    biomarkers.append({'vowel': vowel.split('(')[0].strip(), **{k:v for k,v in bm.items()
                                                                  if not hasattr(v,'__len__')}})
    ax_bm = fig.add_subplot(gs_fig[2, col])
    ax_bm.axis('off')
    txt = (f"f₀={f0_v}Hz\n"
           f"Jitter={bm['jitter_pct']:.2f}%\n"
           f"Shimmer={bm['shimmer_dB']:.2f}dB\n"
           f"HNR={bm['HNR_dB']:.1f}dB")
    ax_bm.text(0.1, 0.9, txt, transform=ax_bm.transAxes,
               va='top', fontsize=9, fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Biotech Singer: LF Glottal Model · Formant Filter · Jitter/Shimmer Biomarkers',
             fontsize=11, fontweight='bold')
plt.show()

# ── Biomarker table ───────────────────────────────────────────────
print('\nVoice biomarker table:')
print(f'{"Vowel":10}  {"Jitter%":9}  {"Shimmer dB":11}  {"HNR dB":8}  Clinical flag')
print('='*60)
for bm in biomarkers:             # loop over vowels
    j = bm.get('jitter_pct', np.nan)
    s = bm.get('shimmer_dB', np.nan)
    h = bm.get('HNR_dB', np.nan)
    flag = []
    if not np.isnan(j) and j > 1.0:  flag.append('jitter↑')
    if not np.isnan(s) and s > 0.35: flag.append('shimmer↑')
    if not np.isnan(h) and h < 7.0:  flag.append('HNR↓ (breathy)')
    flag_str = ', '.join(flag) if flag else 'Normal'
    print(f'{bm["vowel"]:10}  {j:9.3f}  {s:11.3f}  {h:8.1f}  {flag_str}')

In [ ]:
# ── Experiment 7: Dispersion-Assisted Acoustic Measurement ──────
# Connect the repo's GS phase recovery to room acoustics measurement
# Chirped acoustic pulse → propagate through room → stretch & measure spectrum

def chirped_acoustic_pulse(t, f_start=200, f_end=8000, T_sweep=0.05):
    """Linear chirp: sweeps from f_start to f_end over T_sweep seconds."""
    k_rate = (f_end - f_start) / T_sweep
    return np.sin(2*np.pi*(f_start*t + k_rate/2*t**2))

def room_impulse_response(t, RT60, c=343.0, room_vol=200.0, n_modes=20):
    """
    Simulate room IR as sum of decaying modal resonances.
    RT60 determined by Sabine: RT60 = 0.161*V/(alpha*S)
    """
    sr_local = 1.0/(t[1]-t[0])
    decay    = np.exp(-6.91 * t / RT60) * (t >= 0)
    h        = decay * np.random.randn(len(t))
    # Add discrete modes
    L_room = room_vol**(1/3)
    for n in range(1, n_modes+1):         # loop: room modes
        f_mode = n * c / (2*L_room)
        if f_mode < sr_local/2:
            h += 0.3/n * np.exp(-6.91*t/RT60) * np.sin(2*np.pi*f_mode*t) * (t>=0)
    return h / (np.max(np.abs(h)) + 1e-9)

def gs_phase_recover_1d(I_target, n_iter=80, D=-5000, dt_gs=1e-6):
    """
    1D GS phase recovery with dispersion D (as in repo gs_core.py).
    Mimics the chirp-based diversity from the repo.
    """
    A = np.sqrt(np.clip(I_target, 0, None))
    np.random.seed(0)
    E = A * np.exp(1j * 2*np.pi*np.random.rand(len(A)))
    N = len(A)
    omega = 2*np.pi*np.fft.fftfreq(N, dt_gs)
    H_D = np.exp(1j * D/2 * omega**2 * dt_gs**2)  # dispersion diversity
    errs = []
    for it in range(n_iter):              # GS loop
        E    = A * np.exp(1j*np.angle(E))
        E_f  = np.fft.fft(E) * H_D
        E    = np.fft.ifft(E_f)
        errs.append(np.sqrt(np.mean((np.abs(E)-A)**2)))
    return np.angle(E), np.array(errs)

# Generate experiment
sr_ac  = 44100
t_ac   = np.linspace(0, 0.5, int(sr_ac*0.5))

# Room scenarios (concert hall, practice room, anechoic)
rooms  = [('Anechoic',    0.05), ('Practice room', 0.4),
           ('Studio',     0.8),  ('Concert hall',  2.0)]

fig, axes = plt.subplots(len(rooms), 4, figsize=(16, 12))

for row, (room_name, RT60) in enumerate(rooms):  # loop: room types
    # Source chirp
    s_chirp = chirped_acoustic_pulse(t_ac)

    # Room impulse response
    h_room = room_impulse_response(t_ac, RT60)

    # Convolve
    y_room = np.real(np.fft.ifft(np.fft.fft(s_chirp) * np.fft.fft(h_room)))
    y_room += 0.01*np.random.randn(len(y_room))

    # Measured intensity (magnitude spectrum squared)
    I_meas = np.abs(np.fft.fft(y_room))**2

    # GS phase recovery
    phi_rec, errs = gs_phase_recover_1d(I_meas, n_iter=60, D=-5000, dt_gs=1/sr_ac)

    # Reconstruct
    E_rec = np.sqrt(I_meas) * np.exp(1j*phi_rec)
    y_rec = np.real(np.fft.ifft(E_rec))

    # Plots
    t_ms = t_ac * 1000
    axes[row,0].plot(t_ms[:1000], y_room[:1000], 'b-', lw=0.8)
    axes[row,0].set_ylabel(f'{room_name}\nRT60={RT60}s', fontsize=8, rotation=0, labelpad=60)
    if row==0: axes[row,0].set_title('Received signal', fontsize=9)
    axes[row,0].tick_params(labelsize=7)

    freqs_ac = np.fft.rfftfreq(len(t_ac), 1/sr_ac)
    axes[row,1].semilogy(freqs_ac, I_meas[:len(freqs_ac)], 'g-', lw=0.8)
    axes[row,1].set_xlim(0, 8000)
    if row==0: axes[row,1].set_title('|Ẽ(ω)|² measured', fontsize=9)
    axes[row,1].tick_params(labelsize=7)

    axes[row,2].plot(t_ms[:1000], y_rec[:1000], 'r-', lw=0.8, alpha=0.8)
    axes[row,2].plot(t_ms[:1000], s_chirp[:1000]*0.3, 'k--', lw=0.6, alpha=0.4)
    if row==0: axes[row,2].set_title('GS reconstructed', fontsize=9)
    axes[row,2].tick_params(labelsize=7)

    axes[row,3].semilogy(errs, 'purple', lw=1.5)
    if row==0: axes[row,3].set_title('GS convergence', fontsize=9)
    axes[row,3].set_xlabel('iter', fontsize=7); axes[row,3].tick_params(labelsize=7)
    axes[row,3].grid(True, alpha=0.3)

plt.suptitle('Dispersion-Assisted GS Phase Recovery for Room Acoustic Measurement\n'
             '(Repo: ColinsCoding/Dispersion-Assisted-GS-Phase-Recovery)',
             fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nExperiment summary for biotech singer:')
print('1. Record /aː/ vowel → jitter/shimmer → vocal health marker')
print('2. LDV on larynx surface → vocal cord displacement spectrum')
print('3. Photoacoustic measurement → sub-glottal pressure via breath CO₂')
print('4. Chirped acoustic pulse in practice room → GS recovery → RT60')
print('5. Compare singer vs non-singer HNR distribution (Torch batch t-test)')